# 04C — SIMCA pure test evaluation

Goal: evaluate SIMCA candidate configurations on an external pure test batch.

Protocol:

- hyperparameters were selected on validation batch 3 in notebooks `04A`, `04B` and optionally `04B2`;
- this notebook evaluates candidate configurations on pure batch 4;
- batch 4 is not used for hyperparameter optimization;
- mixture images are not used here.

Recommended protocol:

- training for pure test: pure peanut objects from batches 1, 2 and 3;
- projection/test: pure almond and pure peanut objects from batch 4.

Main outputs:

- `pure_test_metrics.parquet`
- `pure_test_model_summary.parquet`
- `pure_test_object_predictions.parquet`
- `pure_test_object_errors_by_image.parquet`
- `pure_test_pixel_errors_by_image.parquet`
- `frozen_reference_configs.parquet`
- `pure_test_protocol.parquet`

The frozen configurations are then used in `05_simca_mixture_application.ipynb`.


In [1]:
from __future__ import annotations

import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 260)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
from src.io.database_h5 import load_nir_uco_h5
from src import experiment_config as expcfg

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
    parse_preprocessing_steps,
    merge_config_metadata,
)

from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs

from src.decision.metrics import (
    add_binary_confusion_case,
    summarize_object_errors_by_image,
    summarize_pixel_errors_by_image,
)

from src.decision.border import (
    summarize_border_diagnostics_by_config,
)

from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
)

from src.workflows.simca_selection_utils import (
    ensure_candidate_columns,
    normalize_simca_rule_columns,
    fill_selected_config_defaults,
    add_detection_selection_score,
)

from src.decision.uncertainty import (
    evaluate_three_way_by_config,
)

from src.visualization.plot_model_selection import (
    plot_validation_test_shift,
)

from src.visualization.plot_robustness import (
    plot_border_core_metrics,
)

from src.visualization.plot_reporting import (
    plot_per_image_performance,
)

from src.visualization.tables import (
    build_frozen_reference_table,
)

%load_ext autoreload
%autoreload 2


## 1. Configuration

This notebook first looks for the candidate file produced by `04B2_optuna_challenge_optional.ipynb`.

If Optuna has not been run, it falls back to the robust candidates produced by `04B_simca_validation_robustness.ipynb`.


In [3]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG

# ---------------------------------------------------------------------
# Inputs from 04B / 04B2
# ---------------------------------------------------------------------
RESULTS_04B_DIR = PROJECT_ROOT / "results" / f"04B_simca_validation_robustness_{RESULTS_TAG}"
RESULTS_04B2_DIR = PROJECT_ROOT / "results" / f"04B2_optuna_challenge_{RESULTS_TAG}"

ROBUST_CANDIDATE_CONFIGS_PATH = RESULTS_04B_DIR / "04B_final_candidate_panel.parquet"

OPTUNA_CANDIDATES_FOR_TEST_PATH = (
    RESULTS_04B2_DIR
    / "candidate_configs_for_pure_test.parquet"
)

# Prefer the combined 04B + 04B2 candidate file when available.
if OPTUNA_CANDIDATES_FOR_TEST_PATH.exists():
    CANDIDATE_CONFIGS_PATH = OPTUNA_CANDIDATES_FOR_TEST_PATH
    CANDIDATE_SOURCE = "04B2_grid_plus_optuna"
else:
    CANDIDATE_CONFIGS_PATH = ROBUST_CANDIDATE_CONFIGS_PATH
    CANDIDATE_SOURCE = "04B_robust_grid_only"

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"04C_simca_pure_test_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PURE_TEST_3WAY_OBJECTS_PATH = RESULTS_DIR / "pure_test_3way_objects.parquet"
PURE_TEST_3WAY_METRICS_PATH = RESULTS_DIR / "pure_test_3way_metrics.parquet"
PURE_TEST_3WAY_BY_IMAGE_PATH = RESULTS_DIR / "pure_test_3way_by_image.parquet"
PURE_TEST_METRICS_PATH = RESULTS_DIR / "pure_test_metrics.parquet"
PURE_TEST_MODEL_SUMMARY_PATH = RESULTS_DIR / "pure_test_model_summary.parquet"
PURE_TEST_OBJECT_PREDICTIONS_PATH = RESULTS_DIR / "pure_test_object_predictions.parquet"
PURE_TEST_OBJECT_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "pure_test_object_errors_by_image.parquet"
PURE_TEST_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "pure_test_pixel_errors_by_image.parquet"

PURE_TEST_BORDER_DIAGNOSTIC_PATH = RESULTS_DIR / "pure_test_border_diagnostic.parquet"
PURE_TEST_REFIT_ERRORS_PATH = RESULTS_DIR / "pure_test_refit_errors.parquet"

FROZEN_REFERENCE_CONFIGS_PATH = RESULTS_DIR / "frozen_reference_configs.parquet"
PURE_TEST_PROTOCOL_PATH = RESULTS_DIR / "pure_test_protocol.parquet"

# Optional debug output. Keep False by default to avoid large files.
SAVE_PURE_TEST_PIXEL_TABLE = False
PURE_TEST_PIXEL_PREDICTIONS_PATH = RESULTS_DIR / "debug" / "pure_test_pixel_predictions.parquet"

# ---------------------------------------------------------------------
# Detection protocol
# ---------------------------------------------------------------------
TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = expcfg.REFERENCE_CLASSES

# Hyperparameters are already selected. For the external pure test, the
# final calibration uses train + validation target data.
PURE_TEST_TRAIN_BATCHES = list(expcfg.PURE_TEST_TRAIN_BATCHES)

PURE_TEST_TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=PURE_TEST_TRAIN_BATCHES,
)

PURE_TEST_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": list(expcfg.PURE_TEST_BATCHES),
}

# ---------------------------------------------------------------------
# Runtime settings
# ---------------------------------------------------------------------
RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS

CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

# ---------------------------------------------------------------------
# Pure-test guardrails
# ---------------------------------------------------------------------
# These thresholds are only external sanity checks.
# They must not be used to tune hyperparameters.
USE_TEST_GUARDRAILS = True

# Binary 2-way guardrails.
MAX_PURE_TEST_BINARY_FN_RATE = 0.35
MAX_PURE_TEST_BINARY_FP_RATE = 0.80
MIN_PURE_TEST_BINARY_BALANCED_ACCURACY = 0.50

# Three-way screening guardrails.
# In 3-way mode:
# - target or uncertain = kept / inspected
# - non_target = rejected
MAX_PURE_TEST_3WAY_TARGET_MISS_RATE = 0.35
MAX_PURE_TEST_3WAY_FALSE_ACCEPT_RATE = 0.80
MAX_PURE_TEST_3WAY_UNCERTAIN_RATE = 0.90

# Final frozen panel size.
# Keep families and candidate sources separated.
N_FROZEN_PER_MATRIX_FAMILY = 8
N_FROZEN_PER_MATRIX_FAMILY_SOURCE = 4
N_FROZEN_OVERALL = None

PRESERVE_CANDIDATE_SOURCES = True

# Border/core diagnostic on pure test.
RUN_BORDER_DIAGNOSTIC = True
BORDER_DIAGNOSTIC_WIDTHS = [1, 2, 3]
BORDER_DIAGNOSTIC_CONFIG_LIMIT = 30

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("CANDIDATE_CONFIGS_PATH:", CANDIDATE_CONFIGS_PATH)
print("CANDIDATE_SOURCE:", CANDIDATE_SOURCE)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("RESULTS_TAG:", RESULTS_TAG)
print("PURE_TEST_TRAIN_FILTERS:", PURE_TEST_TRAIN_FILTERS)
print("PURE_TEST_FILTERS:", PURE_TEST_FILTERS)


DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all
CANDIDATE_CONFIGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B2_optuna_challenge_non_noisy_all\candidate_configs_for_pure_test.parquet
CANDIDATE_SOURCE: 04B2_grid_plus_optuna
WAVELENGTH_MODE: non_noisy_all
RESULTS_TAG: non_noisy_all
PURE_TEST_TRAIN_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['peanut'], 'batch': [1, 2, 3]}
PURE_TEST_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['almond', 'peanut'], 'batch': [4]}


## 2. Load database and candidate configurations

In [4]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

if not CANDIDATE_CONFIGS_PATH.exists():
    raise FileNotFoundError(
        f"Candidate configs not found: {CANDIDATE_CONFIGS_PATH}. "
        "Run 04B first, and optionally 04B2."
    )

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

candidate_configs_df = load_parquet(CANDIDATE_CONFIGS_PATH)

candidate_configs_df = ensure_candidate_columns(candidate_configs_df)
candidate_configs_df = normalize_simca_rule_columns(candidate_configs_df)
candidate_configs_df = fill_selected_config_defaults(
    candidate_configs_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)

candidate_configs_df = candidate_configs_df.copy()
candidate_configs_df["selected_config_id"] = candidate_configs_df["selected_config_id"].astype(str)
candidate_configs_df["candidate_source_file"] = str(CANDIDATE_CONFIGS_PATH)
candidate_configs_df["candidate_file_source_kind"] = CANDIDATE_SOURCE

if "candidate_source" not in candidate_configs_df.columns:
    candidate_configs_df["candidate_source"] = CANDIDATE_SOURCE
else:
    candidate_configs_df["candidate_source"] = (
        candidate_configs_df["candidate_source"]
        .fillna(CANDIDATE_SOURCE)
        .astype(str)
    )

candidate_configs_df["candidate_input_rank"] = np.arange(
    1,
    len(candidate_configs_df) + 1,
)

required_candidate_cols = [
    "selected_config_id",
    "matrix_family",
    "matrix_method",
    "preprocessing",
    "preprocessing_steps",
    "rule_for_refit",
    "n_components",
    "alpha",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_candidate_cols = [
    col for col in required_candidate_cols
    if col not in candidate_configs_df.columns
]

if missing_candidate_cols:
    raise KeyError(
        "Candidate config file is missing required columns for 04C: "
        f"{missing_candidate_cols}. "
        "Re-run 04B and 04B2 with fixed 3-way threshold export."
    )

if candidate_configs_df["selected_config_id"].duplicated().any():
    duplicates = candidate_configs_df.loc[
        candidate_configs_df["selected_config_id"].duplicated(keep=False),
        "selected_config_id",
    ].unique()
    raise ValueError(f"Duplicated selected_config_id values: {duplicates[:10]}")

print("Database loaded")
print("n objects:", len(object_db))
print("n images:", len(image_db))
print("n active bands:", len(wavelengths))
print("Candidate configs:", candidate_configs_df.shape)

print("Candidate sources:")
display(candidate_configs_df["candidate_source"].value_counts(dropna=False))

print("Matrix families:")
display(candidate_configs_df["matrix_family"].value_counts(dropna=False))

display(wavelength_config_df)
display(candidate_configs_df.head(20))

Database loaded
n objects: 1262
n images: 48
n active bands: 63
Candidate configs: (31, 104)
Candidate sources:


candidate_source
04B_grid_robustness      18
04B2_optuna_challenge    13
Name: count, dtype: int64

Matrix families:


matrix_family
object_matrix    20
pixel_matrix     11
Name: count, dtype: int64

,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,selected_config_id,robust_selection_rank,selection_split,selection_strategy,robust_pareto_axis,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,three_way_lower_threshold,three_way_upper_threshold,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,is_robust_2way_pareto,is_robust_3way_pareto,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,threeway_mean_target_miss_rate,threeway_max_target_miss_rate,threeway_mean_non_target_false_accept_rate,threeway_mean_uncertain_rate,threeway_std_uncertain_rate,threeway_mean_coverage_rate,rule_original,rule_variant_original,rule_token,candidate_source,number,state,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,value,n_components,balanced_accuracy_mean,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_max,fp_rate_mean,fp_rate_std,object_threshold_median,matrix_family_study,optuna_trial_number,fn_rate,fn_rate_source,fp_rate,fp_rate_source,balanced_accuracy,balanced_accuracy_source,target_sensitivity,target_sensitivity_source,non_target_specificity,non_target_specificity_source,f1_score,f1_score_source,accuracy,accuracy_source,precision,precision_source,selection_score,optuna_score_conservative_target,optuna_score_balanced_reference,optuna_score_specificity_control,validation_f1_score,validation_accuracy,validation_precision,candidate_source_file,candidate_file_source_kind,candidate_input_rank
0,optuna_object_matrix_0149,NaN,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated,NaN,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,almond,0.01,0.50,15,2,4,0.50,0.95,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.345455,0.509259,0.490741,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,04B2_optuna_challenge,149.0,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,5,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.50,object_matrix,149.0,0.000000,fn_rate_max,0.927273,fp_rate_mean,0.536364,balanced_accuracy_mean,1.000000,None,0.072727,None,NaN,None,NaN,None,NaN,None,-0.927273,-1.318182,0.681818,-2.563636,0.675159,0.527778,0.509615,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,1
1,optuna_object_matrix_0081,NaN,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated,NaN,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,0.01,0.50,15,2,5,0.50,0.95,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.290909,0.546296,0.453704,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,data_driven,data_driven_emp_cv,data_driven_emp_cv,04B2_optuna_challenge,81.0,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,5,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.50,object_matrix,81.0,0.000000,fn_rate_max,0.927273,fp_rate_mean,0.536364,balanced_accuracy_mean,1.000000,None,0.072727,None,NaN,None,NaN,None,NaN,None,-0.927273,-1.318182,0.681818,-2.563636,0.675159,0.527778,0.509615,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,2
2,optuna_object_matrix_0148,NaN,vali

## 3. Rebuild preprocessing configurations

In [5]:
required_cols = ["preprocessing", "preprocessing_steps"]
missing = [col for col in required_cols if col not in candidate_configs_df.columns]
if missing:
    raise KeyError(f"Missing required candidate columns: {missing}")

PREPROCESSING_CONFIGS = {
    str(row["preprocessing"]): tuple(
        parse_preprocessing_steps(
            row["preprocessing_steps"]
        )
    )
    for _, row in (
        candidate_configs_df
        .drop_duplicates("preprocessing")
        .iterrows()
    )
}

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

preprocessing_configs_df = pd.DataFrame([
    {
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
    }
    for name, steps in PREPROCESSING_CONFIGS.items()
])

print("Preprocessing configs used for pure test refit:")
display(preprocessing_configs_df)

print("Number of preprocessing configs:", len(PREPROCESSING_CONFIGS))

Preprocessing configs used for pure test refit:


,preprocessing,preprocessing_steps
0,sg_smooth,sg_smooth
1,absorbance_sg_smooth,absorbance+sg_smooth
2,absorbance_sg_d2,absorbance+sg_d2
3,snv_sg_smooth,snv+sg_smooth
4,raw,raw
5,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth
6,absorbance_snv_sg_d1,absorbance+snv+sg_d1


Number of preprocessing configs: 7


## 4. External pure test refit and projection

The model configurations are fixed before this step. Here we only refit the selected one-class peanut SIMCA models on pure peanut batches 1, 2 and 3, then evaluate them on pure batch 4.


In [6]:
CANDIDATE_METADATA_COLS = [
    "candidate_source",
    "candidate_source_file",
    "candidate_file_source_kind",
    "candidate_input_rank",
    "selection_split",
    "selection_strategy",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
    "validation_fn_rate",
    "validation_fp_rate",
    "validation_3way_target_miss_rate",
    "validation_3way_non_target_false_accept_rate",
    "validation_3way_uncertain_rate",
    "validation_3way_coverage_rate",
]

In [7]:
(
    pure_test_metrics_df,
    pure_test_objects_df,
    pure_test_pixels_df,
    pure_test_pixel_errors_by_image_df,
    pure_test_refit_errors_df,
) = refit_selected_simca_configs(
    selected_configs_df=candidate_configs_df,
    object_db=object_db,
    image_db=image_db,
    train_filters=PURE_TEST_TRAIN_FILTERS,
    projection_filters=PURE_TEST_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    evaluation_split="pure_test_batch_4",
    wavelengths=wavelengths,
    random_state=RANDOM_STATE,
    replace=REPLACE_BALANCED_PIXELS,
    cv_n_splits=CV_N_SPLITS,
    cv_group_col=CV_GROUP_COL,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

pure_test_objects_df = add_binary_confusion_case(
    pure_test_objects_df,
    target_class=TARGET_CLASS,
    level="object",
)

pure_test_pixels_df = add_binary_confusion_case(
    pure_test_pixels_df,
    target_class=TARGET_CLASS,
    level="pixel",
)

pure_test_objects_df = merge_config_metadata(
    pure_test_objects_df,
    candidate_configs_df,
    id_col="selected_config_id",
    columns=CANDIDATE_METADATA_COLS,
)

pure_test_pixels_df = merge_config_metadata(
    pure_test_pixels_df,
    candidate_configs_df,
    id_col="selected_config_id",
    columns=CANDIDATE_METADATA_COLS,
)

pure_test_metrics_df = add_detection_selection_score(
    pure_test_metrics_df,
    score_col="pure_test_selection_score",
)

save_parquet(pure_test_metrics_df, PURE_TEST_METRICS_PATH)
save_parquet(pure_test_objects_df, PURE_TEST_OBJECT_PREDICTIONS_PATH)
save_parquet_if_nonempty(pure_test_refit_errors_df, PURE_TEST_REFIT_ERRORS_PATH)

if SAVE_PURE_TEST_PIXEL_TABLE:
    save_parquet(pure_test_pixels_df, PURE_TEST_PIXEL_PREDICTIONS_PATH)

print("Pure test metrics:", pure_test_metrics_df.shape)
print("Pure test objects:", pure_test_objects_df.shape)
print("Pure test pixels:", pure_test_pixels_df.shape)
print("Pure test refit errors:", pure_test_refit_errors_df.shape)

display(pure_test_metrics_df.head(30))
display(pure_test_refit_errors_df)


[pure_test_batch_4] optuna_object_matrix_0149
[pure_test_batch_4] optuna_object_matrix_0081
[pure_test_batch_4] optuna_object_matrix_0148
[pure_test_batch_4] optuna_object_matrix_0184
[pure_test_batch_4] optuna_object_matrix_0152
[pure_test_batch_4] optuna_object_matrix_0157
[pure_test_batch_4] optuna_object_matrix_0105
[pure_test_batch_4] optuna_object_matrix_0121
[pure_test_batch_4] 04A_object_matrix_0001
[pure_test_batch_4] 04A_object_matrix_0002
[pure_test_batch_4] 04A_object_matrix_0003
[pure_test_batch_4] 04A_object_matrix_0004
[pure_test_batch_4] 04A_object_matrix_0005
[pure_test_batch_4] 04A_object_matrix_0006
[pure_test_batch_4] 04A_object_matrix_0007
[pure_test_batch_4] 04A_object_matrix_0008
[pure_test_batch_4] 04A_object_matrix_0009
[pure_test_batch_4] 04A_object_matrix_0010
[pure_test_batch_4] 04A_object_matrix_0012
[pure_test_batch_4] 04A_object_matrix_0015
[pure_test_batch_4] optuna_pixel_matrix_0111
[pure_test_batch_4] optuna_pixel_matrix_0285
[pure_test_batch_4] optuna

,selected_config_id,robust_selection_rank,selection_split,selection_strategy,robust_pareto_axis,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,three_way_lower_threshold,three_way_upper_threshold,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,is_robust_2way_pareto,is_robust_3way_pareto,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,threeway_mean_target_miss_rate,threeway_max_target_miss_rate,threeway_mean_non_target_false_accept_rate,threeway_mean_uncertain_rate,threeway_std_uncertain_rate,threeway_mean_coverage_rate,rule_original,rule_variant_original,rule_token,candidate_source,number,state,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,value,n_components,balanced_accuracy_mean,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_max,fp_rate_mean,fp_rate_std,object_threshold_median,matrix_family_study,optuna_trial_number,fn_rate,fn_rate_source,fp_rate,fp_rate_source,balanced_accuracy,balanced_accuracy_source,target_sensitivity,target_sensitivity_source,non_target_specificity,non_target_specificity_source,f1_score,f1_score_source,accuracy,accuracy_source,precision,precision_source,selection_score,optuna_score_conservative_target,optuna_score_balanced_reference,optuna_score_specificity_control,validation_f1_score,validation_accuracy,validation_precision,candidate_source_file,candidate_file_source_kind,candidate_input_rank,non_target_class,n,tp,fn,fp,tn,evaluation_split,n_projected_objects,n_projected_pixels,pure_test_selection_score
0,optuna_object_matrix_0149,NaN,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated,NaN,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,almond,0.01,0.50,15,2,4,0.50,0.95,0.536364,1.000000,0.072727,0.000000,0.927273,0.000000,1.000000,0.345455,0.509259,0.490741,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,04B2_optuna_challenge,149.0,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,5.0,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.50,object_matrix,149.0,0.000000,fn_rate_max,0.437500,fp_rate_mean,0.781250,balanced_accuracy_mean,1.000000,None,0.562500,None,0.734177,None,0.727273,None,0.580000,None,-0.927273,-1.318182,0.681818,-2.563636,0.675159,0.527778,0.509615,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,1,almond,77,29,0,21,27,pure_test_batch_4,77,8401,-0.386246
1,optuna_object_matrix_0081,NaN,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated,NaN,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,0.01,0.50,15,2,5,0.50,0.95,0.536364,1.000000,0.072727,0.000000,0.927273,0.000000,1.000000,0.290909,0.546296,0.453704,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,data_driven,data_driven_emp_cv,data_driven_emp_cv,04B2_optuna_challenge,81.0,COMPLETE,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,NaN,5.0,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.50,object_matrix,81.0,0.000000,fn_rate_max,0.437500,fp_rate_mean,0.781250,balanced_accuracy_mean,1.000000,None,0.562500

""


In [8]:
# ---------------------------------------------------------------------
# Apply fixed 3-way thresholds on pure test batch 4
# ---------------------------------------------------------------------
# No calibration is performed here.
# The lower/upper thresholds come from 04B or 04B2.

pure_test_3way_metrics_df, pure_test_3way_objects_df = evaluate_three_way_by_config(
    object_df=pure_test_objects_df,
    thresholds_df=candidate_configs_df,
    config_id_col="selected_config_id",
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

pure_test_3way_by_image_df, _ = evaluate_three_way_by_config(
    object_df=pure_test_objects_df,
    thresholds_df=candidate_configs_df,
    config_id_col="selected_config_id",
    extra_group_cols=["source_image"],
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

pure_test_3way_objects_df = merge_config_metadata(
    pure_test_3way_objects_df,
    candidate_configs_df,
    id_col="selected_config_id",
    columns=CANDIDATE_METADATA_COLS,
)

save_parquet(pure_test_3way_metrics_df, PURE_TEST_3WAY_METRICS_PATH)
save_parquet(pure_test_3way_objects_df, PURE_TEST_3WAY_OBJECTS_PATH)
save_parquet_if_nonempty(pure_test_3way_by_image_df, PURE_TEST_3WAY_BY_IMAGE_PATH)

print("Pure test 3-way metrics:", pure_test_3way_metrics_df.shape)
print("Pure test 3-way objects:", pure_test_3way_objects_df.shape)
print("Pure test 3-way by image:", pure_test_3way_by_image_df.shape)

display(
    pure_test_3way_metrics_df[
        [
            col for col in [
                "selected_config_id",
                "n",
                "n_target",
                "n_non_target",
                "target_miss_rate",
                "screening_sensitivity",
                "non_target_false_accept_rate",
                "uncertain_rate",
                "coverage_rate",
                "non_target_auto_reject_rate",
                "decided_balanced_accuracy",
            ]
            if col in pure_test_3way_metrics_df.columns
        ]
    ].head(30)
)

Pure test 3-way metrics: (31, 21)
Pure test 3-way objects: (2387, 66)
Pure test 3-way by image: (62, 22)


,selected_config_id,n,n_target,n_non_target,target_miss_rate,screening_sensitivity,non_target_false_accept_rate,uncertain_rate,coverage_rate,non_target_auto_reject_rate,decided_balanced_accuracy
0,04A_object_matrix_0001,77,29,48,0.000000,1.000000,0.000000,0.675325,0.324675,0.520833,NaN
1,04A_object_matrix_0002,77,29,48,0.034483,0.965517,0.000000,0.857143,0.142857,0.187500,0.750000
2,04A_object_matrix_0003,77,29,48,0.034483,0.965517,0.000000,0.857143,0.142857,0.187500,0.750000
3,04A_object_matrix_0004,77,29,48,0.034483,0.965517,0.000000,0.818182,0.181818,0.250000,0.750000
4,04A_object_matrix_0005,77,29,48,0.034483,0.965517,0.000000,0.818182,0.181818,0.250000,0.750000
5,04A_object_matrix_0006,77,29,48,0.000000,1.000000,0.000000,0.792208,0.207792,0.291667,1.000000
6,04A_object_matrix_0007,77,29,48,0.000000,1.000000,0.062500,0.714286,0.285714,0.250000,0.900000
7,04A_object_matrix_0008,77,29,48,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.500000
8,04A_object_matrix_0009,77,29,48,0.965517,0.034483,0.000000,0.012987,0.987013,1.000000,0.500000
9,04A_object_matrix_0010,77,29,48,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.500000


## 5. Error summaries by image

In [9]:
MODEL_GROUP_COLS = [
    "selected_config_id",
    "candidate_source",
    "selection_strategy",
    "matrix_family",
    "training_matrix_id",
    "model_family",
    "matrix_method",
    "preprocessing",
    "selected_rule_name",
    "rule",
    "rule_variant",
    "rule_for_refit",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "m",
    "m_effective",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
]

OBJECT_IMAGE_GROUP_COLS = [
    col for col in MODEL_GROUP_COLS + ["source_image"]
    if col in pure_test_objects_df.columns
]

PIXEL_IMAGE_GROUP_COLS = [
    col for col in MODEL_GROUP_COLS + ["source_image"]
    if col in pure_test_pixels_df.columns
]

pure_test_object_errors_by_image_df = summarize_object_errors_by_image(
    pure_test_objects_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=OBJECT_IMAGE_GROUP_COLS,
)

pure_test_pixel_errors_by_image_df = summarize_pixel_errors_by_image(
    pure_test_pixels_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=PIXEL_IMAGE_GROUP_COLS,
)

save_parquet(pure_test_object_errors_by_image_df, PURE_TEST_OBJECT_ERRORS_BY_IMAGE_PATH)
save_parquet(pure_test_pixel_errors_by_image_df, PURE_TEST_PIXEL_ERRORS_BY_IMAGE_PATH)

print("Object errors by image:", pure_test_object_errors_by_image_df.shape)
print("Pixel errors by image:", pure_test_pixel_errors_by_image_df.shape)

display(pure_test_object_errors_by_image_df.head(30))
display(pure_test_pixel_errors_by_image_df.head(30))


Object errors by image: (62, 43)
Pixel errors by image: (62, 43)


,selected_config_id,candidate_source,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,source_image,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_objects,object_accuracy,object_balanced_accuracy,object_fn_rate,object_fp_rate
0,04A_object_matrix_0001,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,absorbance_sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,0,29,0,0,0.000000,NaN,NaN,0.000000,NaN,NaN,1.000000,NaN,29,0.000000,NaN,1.000000,NaN
1,04A_object_matrix_0008,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,0,29,0,0,0.000000,NaN,NaN,0.000000,NaN,NaN,1.000000,NaN,29,0.000000,NaN,1.000000,NaN
2,04A_object_matrix_0009,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,simple_emp_cv,simple,simple_emp_cv,simple_emp_cv,NaN,0.01,0.80,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,0,29,0,0,0.000000,NaN,NaN,0.000000,NaN,NaN,1.000000,NaN,29,0.000000,NaN,1.000000,NaN
3,04A_object_matrix_0010,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.80,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,0,29,0,0,0.000000,NaN,NaN,0.000000,NaN,NaN,1.000000,NaN,29,0.000000,NaN,1.000000,NaN
4,optuna_pixel_matrix_0273,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_d2,simple_chi2,simple,simple_chi2,simple_chi2,10.0,0.05,0.50,13,2,3,20,NaN,center,NaN,peanut4,peanut,almond,29,9,20,0,0,0.310345,NaN,NaN,0.310345,1.0,0.473684,0.689655,NaN,29,0.310345,NaN,0.689655,NaN
5,04A_object_matrix_0006,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_median,empirical_cv_rule,object_median,raw,alternative_chi2_emp_cv,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,12,17,0,0,0.413793,NaN,NaN,0.413793,1.0,0.585366,0.586207,NaN,29,0.413793,NaN,0.586207,NaN
6,04A_object_matrix_0015,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_median,empirical_cv_rule,object_median,snv_sg_smooth,data_driven_chi2,data_driven,data_driven_chi2,data_driven_chi2,NaN,0.01,0.70,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,15,14,0,0,0.517241,NaN,NaN,0.517241,1.0,0.681818,0.482759,NaN,29,0.517241,NaN,0.482759,NaN
7,04A_object_matrix_0005,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,simple_emp_cv,simple,simple_emp_cv,simple_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,29,16,13,0,0,0.551724,NaN,NaN,0.551724,1.0,0.711111,0.448276,NaN,29,0.551724,NaN,0.448276,NaN
8,optuna_object_matrix_0157,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,combined_index_chi2,combined_index,combined_index_chi2,combined_index_chi2,3.0,0.01,0.50,15,2,5,40,NaN,random,NaN,peanut4,

,selected_config_id,candidate_source,selection_strategy,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,source_image,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_pixels,pixel_accuracy,pixel_balanced_accuracy,pixel_fn_rate,pixel_fp_rate
0,04A_object_matrix_0008,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,3164,335,2829,0,0,0.105879,NaN,NaN,0.105879,1.0,0.191483,0.894121,NaN,3164,0.105879,NaN,0.894121,NaN
1,04A_object_matrix_0010,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.80,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,3164,335,2829,0,0,0.105879,NaN,NaN,0.105879,1.0,0.191483,0.894121,NaN,3164,0.105879,NaN,0.894121,NaN
2,04A_object_matrix_0009,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,sg_smooth,simple_emp_cv,simple,simple_emp_cv,simple_emp_cv,NaN,0.01,0.80,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,3164,529,2635,0,0,0.167193,NaN,NaN,0.167193,1.0,0.286488,0.832807,NaN,3164,0.167193,NaN,0.832807,NaN
3,04A_object_matrix_0001,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_mean,empirical_cv_rule,object_mean,absorbance_sg_smooth,data_driven_emp_cv,data_driven,data_driven_emp_cv,data_driven_emp_cv,NaN,0.01,0.75,11,2,3,40,40.0,not_applicable,random,peanut4,peanut,almond,3164,1246,1918,0,0,0.393805,NaN,NaN,0.393805,1.0,0.565079,0.606195,NaN,3164,0.393805,NaN,0.606195,NaN
4,optuna_pixel_matrix_0273,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,pixel_matrix,balanced_pixels,rule_variant_grid,balanced_pixels,absorbance_sg_d2,simple_chi2,simple,simple_chi2,simple_chi2,10.0,0.05,0.50,13,2,3,20,NaN,center,NaN,peanut4,peanut,almond,3164,1375,1789,0,0,0.434576,NaN,NaN,0.434576,1.0,0.605860,0.565424,NaN,3164,0.434576,NaN,0.565424,NaN
5,optuna_object_matrix_0157,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,combined_index_chi2,combined_index,combined_index_chi2,combined_index_chi2,3.0,0.01,0.50,15,2,5,40,NaN,random,NaN,peanut4,peanut,almond,3164,1729,1435,0,0,0.546460,NaN,NaN,0.546460,1.0,0.706724,0.453540,NaN,3164,0.546460,NaN,0.453540,NaN
6,optuna_object_matrix_0105,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,3.0,0.01,0.55,7,2,2,40,NaN,random,NaN,peanut4,peanut,almond,3164,1914,1250,0,0,0.604930,NaN,NaN,0.604930,1.0,0.753840,0.395070,NaN,3164,0.604930,NaN,0.395070,NaN
7,optuna_object_matrix_0121,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,alternative,alternative_empHQ_fixed2,alternative_empHQ_fixed2,3.0,0.01,0.55,11,2,5,40,NaN,random,NaN,peanut4,peanut,almond,3164,1914,1250,0,0,0.604930,NaN,NaN,0.604930,1.0,0.753840,0.395070,NaN,3164,0.604930,NaN,0.395070,NaN
8,04A_object_matrix_0015,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,object_matrix,object_median,empirical_cv_rule,object_median,snv_sg_smooth,data_driven_chi2,data_driven,d

ADD SUMMARY ERRORS BY IMAGE FOR 3WAY !!!!!!!!!!!!!!

In [10]:
if len(pure_test_3way_by_image_df) > 0:
    print("3-way metrics by image:", pure_test_3way_by_image_df.shape)
    display(pure_test_3way_by_image_df.head(30))

3-way metrics by image: (62, 22)


,n,n_target,n_non_target,n_uncertain,uncertain_rate,coverage_rate,target_miss_rate,screening_sensitivity,target_auto_accept_rate,target_uncertain_rate,non_target_false_accept_rate,non_target_auto_reject_rate,non_target_uncertain_rate,decided_tp,decided_fn,decided_fp,decided_tn,decided_accuracy,decided_balanced_accuracy,three_way_score,selected_config_id,source_image
0,48,0,48,23,0.479167,0.520833,NaN,NaN,NaN,NaN,0.000000,0.520833,0.479167,0,0,0,25,1.000000,NaN,-20.109375,04A_object_matrix_0001,almond4
1,29,29,0,29,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,NaN,NaN,NaN,0,0,0,0,NaN,NaN,-2.500000,04A_object_matrix_0001,peanut4
2,48,0,48,39,0.812500,0.187500,NaN,NaN,NaN,NaN,0.000000,0.187500,0.812500,0,0,0,9,1.000000,NaN,-20.359375,04A_object_matrix_0002,almond4
3,29,29,0,27,0.931034,0.068966,0.034483,0.965517,0.034483,0.931034,NaN,NaN,NaN,1,1,0,0,0.500000,NaN,-3.189655,04A_object_matrix_0002,peanut4
4,48,0,48,39,0.812500,0.187500,NaN,NaN,NaN,NaN,0.000000,0.187500,0.812500,0,0,0,9,1.000000,NaN,-20.359375,04A_object_matrix_0003,almond4
5,29,29,0,27,0.931034,0.068966,0.034483,0.965517,0.034483,0.931034,NaN,NaN,NaN,1,1,0,0,0.500000,NaN,-3.189655,04A_object_matrix_0003,peanut4
6,48,0,48,36,0.750000,0.250000,NaN,NaN,NaN,NaN,0.000000,0.250000,0.750000,0,0,0,12,1.000000,NaN,-20.312500,04A_object_matrix_0004,almond4
7,29,29,0,27,0.931034,0.068966,0.034483,0.965517,0.034483,0.931034,NaN,NaN,NaN,1,1,0,0,0.500000,NaN,-3.189655,04A_object_matrix_0004,peanut4
8,48,0,48,36,0.750000,0.250000,NaN,NaN,NaN,NaN,0.000000,0.250000,0.750000,0,0,0,12,1.000000,NaN,-20.312500,04A_object_matrix_0005,almond4
9,29,29,0,27,0.931034,0.068966,0.034483,0.965517,0.034483,0.931034,NaN,NaN,NaN,1,1,0,0,0.500000,NaN,-3.189655,04A_object_matrix_0005,peanut4


In [11]:
plot_per_image_performance(
    pure_test_3way_by_image_df,
    image_col="source_image",
    metric_cols=(
        "target_miss_rate",
        "non_target_false_accept_rate",
        "uncertain_rate",
    ),
    config_col="selected_config_id",
    sort_metric="target_miss_rate",
    worst_first=True,
    top_n=20,
    title="Pure test — 3-way performance by image",
    show=True,
)

## 6. Build pure-test model summary

The table below merges the original candidate metadata with external test metrics. It is useful for interpretation and for flagging catastrophic failures.


In [12]:
# ---------------------------------------------------------------------
# Build pure-test model summary
# ---------------------------------------------------------------------

PURE_TEST_METRIC_COLS = [
    "selected_config_id",
    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "balanced_accuracy",
    "target_sensitivity",
    "non_target_specificity",
    "fn_rate",
    "fp_rate",
    "f1_score",
    "accuracy",
    "precision",
    "pure_test_selection_score",
    "n_projected_objects",
    "n_projected_pixels",
]

PURE_TEST_METRIC_COLS = [
    col for col in PURE_TEST_METRIC_COLS
    if col in pure_test_metrics_df.columns
]

pure_test_metric_prefix_map = {
    "n": "pure_test_n",
    "tp": "pure_test_tp",
    "fn": "pure_test_fn",
    "fp": "pure_test_fp",
    "tn": "pure_test_tn",
    "balanced_accuracy": "pure_test_balanced_accuracy",
    "target_sensitivity": "pure_test_target_sensitivity",
    "non_target_specificity": "pure_test_non_target_specificity",
    "fn_rate": "pure_test_fn_rate",
    "fp_rate": "pure_test_fp_rate",
    "f1_score": "pure_test_f1_score",
    "accuracy": "pure_test_accuracy",
    "precision": "pure_test_precision",
    "n_projected_objects": "pure_test_n_projected_objects",
    "n_projected_pixels": "pure_test_n_projected_pixels",
}

pure_test_metrics_for_merge_df = (
    pure_test_metrics_df[PURE_TEST_METRIC_COLS]
    .rename(columns=pure_test_metric_prefix_map)
)

THREE_WAY_METRIC_COLS = [
    "selected_config_id",
    "n",
    "n_target",
    "n_non_target",
    "n_uncertain",
    "uncertain_rate",
    "coverage_rate",
    "target_miss_rate",
    "screening_sensitivity",
    "target_auto_accept_rate",
    "target_uncertain_rate",
    "non_target_false_accept_rate",
    "non_target_auto_reject_rate",
    "non_target_uncertain_rate",
    "decided_accuracy",
    "decided_balanced_accuracy",
]

THREE_WAY_METRIC_COLS = [
    col for col in THREE_WAY_METRIC_COLS
    if col in pure_test_3way_metrics_df.columns
]

three_way_prefix_map = {
    "n": "pure_test_3way_n",
    "n_target": "pure_test_3way_n_target",
    "n_non_target": "pure_test_3way_n_non_target",
    "n_uncertain": "pure_test_3way_n_uncertain",
    "uncertain_rate": "pure_test_3way_uncertain_rate",
    "coverage_rate": "pure_test_3way_coverage_rate",
    "target_miss_rate": "pure_test_3way_target_miss_rate",
    "screening_sensitivity": "pure_test_3way_screening_sensitivity",
    "target_auto_accept_rate": "pure_test_3way_target_auto_accept_rate",
    "target_uncertain_rate": "pure_test_3way_target_uncertain_rate",
    "non_target_false_accept_rate": "pure_test_3way_non_target_false_accept_rate",
    "non_target_auto_reject_rate": "pure_test_3way_non_target_auto_reject_rate",
    "non_target_uncertain_rate": "pure_test_3way_non_target_uncertain_rate",
    "decided_accuracy": "pure_test_3way_decided_accuracy",
    "decided_balanced_accuracy": "pure_test_3way_decided_balanced_accuracy",
}

pure_test_3way_for_merge_df = (
    pure_test_3way_metrics_df[THREE_WAY_METRIC_COLS]
    .rename(columns=three_way_prefix_map)
)

pure_test_model_summary_df = candidate_configs_df.merge(
    pure_test_metrics_for_merge_df,
    on="selected_config_id",
    how="left",
)

pure_test_model_summary_df = pure_test_model_summary_df.merge(
    pure_test_3way_for_merge_df,
    on="selected_config_id",
    how="left",
)

# ---------------------------------------------------------------------
# Guardrails
# ---------------------------------------------------------------------

pure_test_model_summary_df["passes_pure_test_binary_guardrail"] = (
    (pure_test_model_summary_df["pure_test_fn_rate"].fillna(1.0) <= MAX_PURE_TEST_BINARY_FN_RATE)
    & (pure_test_model_summary_df["pure_test_fp_rate"].fillna(1.0) <= MAX_PURE_TEST_BINARY_FP_RATE)
    & (
        pure_test_model_summary_df["pure_test_balanced_accuracy"].fillna(0.0)
        >= MIN_PURE_TEST_BINARY_BALANCED_ACCURACY
    )
)

pure_test_model_summary_df["passes_pure_test_3way_guardrail"] = (
    (
        pure_test_model_summary_df["pure_test_3way_target_miss_rate"].fillna(1.0)
        <= MAX_PURE_TEST_3WAY_TARGET_MISS_RATE
    )
    & (
        pure_test_model_summary_df["pure_test_3way_non_target_false_accept_rate"].fillna(1.0)
        <= MAX_PURE_TEST_3WAY_FALSE_ACCEPT_RATE
    )
    & (
        pure_test_model_summary_df["pure_test_3way_uncertain_rate"].fillna(1.0)
        <= MAX_PURE_TEST_3WAY_UNCERTAIN_RATE
    )
)

pure_test_model_summary_df["passes_pure_test_guardrail"] = (
    pure_test_model_summary_df["passes_pure_test_binary_guardrail"]
    & pure_test_model_summary_df["passes_pure_test_3way_guardrail"]
)

if not USE_TEST_GUARDRAILS:
    pure_test_model_summary_df["passes_pure_test_binary_guardrail"] = True
    pure_test_model_summary_df["passes_pure_test_3way_guardrail"] = True
    pure_test_model_summary_df["passes_pure_test_guardrail"] = True

pure_test_model_summary_df["pure_test_guardrail_reason"] = "pass"

pure_test_model_summary_df.loc[
    pure_test_model_summary_df["pure_test_fn_rate"].fillna(1.0) > MAX_PURE_TEST_BINARY_FN_RATE,
    "pure_test_guardrail_reason",
] = "high_binary_fn_rate"

pure_test_model_summary_df.loc[
    pure_test_model_summary_df["pure_test_fp_rate"].fillna(1.0) > MAX_PURE_TEST_BINARY_FP_RATE,
    "pure_test_guardrail_reason",
] = "high_binary_fp_rate"

pure_test_model_summary_df.loc[
    (
        pure_test_model_summary_df["pure_test_balanced_accuracy"].fillna(0.0)
        < MIN_PURE_TEST_BINARY_BALANCED_ACCURACY
    ),
    "pure_test_guardrail_reason",
] = "low_binary_balanced_accuracy"

pure_test_model_summary_df.loc[
    (
        pure_test_model_summary_df["pure_test_3way_target_miss_rate"].fillna(1.0)
        > MAX_PURE_TEST_3WAY_TARGET_MISS_RATE
    ),
    "pure_test_guardrail_reason",
] = "high_3way_target_miss_rate"

pure_test_model_summary_df.loc[
    (
        pure_test_model_summary_df["pure_test_3way_non_target_false_accept_rate"].fillna(1.0)
        > MAX_PURE_TEST_3WAY_FALSE_ACCEPT_RATE
    ),
    "pure_test_guardrail_reason",
] = "high_3way_false_accept_rate"

pure_test_model_summary_df.loc[
    (
        pure_test_model_summary_df["pure_test_3way_uncertain_rate"].fillna(1.0)
        > MAX_PURE_TEST_3WAY_UNCERTAIN_RATE
    ),
    "pure_test_guardrail_reason",
] = "high_3way_uncertain_rate"

# Important: do not rank by pure-test metrics.
# Pure test is used only as an external guardrail.
pure_test_model_summary_df = (
    pure_test_model_summary_df
    .sort_values(
        [
            "passes_pure_test_guardrail",
            "candidate_input_rank",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

save_parquet(pure_test_model_summary_df, PURE_TEST_MODEL_SUMMARY_PATH)

print("Pure test model summary:", pure_test_model_summary_df.shape)
print("Saved:", PURE_TEST_MODEL_SUMMARY_PATH)

display(
    pure_test_model_summary_df[
        [
            col for col in [
                "selected_config_id",
                "candidate_source",
                "selection_strategy",
                "candidate_input_rank",
                "matrix_family",
                "training_matrix_id",
                "model_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "three_way_lower_threshold",
                "three_way_upper_threshold",

                "validation_fn_rate",
                "validation_fp_rate",
                "validation_3way_target_miss_rate",
                "validation_3way_uncertain_rate",

                "pure_test_fn_rate",
                "pure_test_fp_rate",
                "pure_test_balanced_accuracy",
                "pure_test_3way_target_miss_rate",
                "pure_test_3way_non_target_false_accept_rate",
                "pure_test_3way_uncertain_rate",
                "pure_test_3way_coverage_rate",

                "passes_pure_test_binary_guardrail",
                "passes_pure_test_3way_guardrail",
                "passes_pure_test_guardrail",
                "pure_test_guardrail_reason",
            ]
            if col in pure_test_model_summary_df.columns
        ]
    ].head(80)
)

Pure test model summary: (31, 139)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_model_summary.parquet


,selected_config_id,candidate_source,selection_strategy,candidate_input_rank,matrix_family,training_matrix_id,model_family,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_uncertain_rate,pure_test_fn_rate,pure_test_fp_rate,pure_test_balanced_accuracy,pure_test_3way_target_miss_rate,pure_test_3way_non_target_false_accept_rate,pure_test_3way_uncertain_rate,pure_test_3way_coverage_rate,passes_pure_test_binary_guardrail,passes_pure_test_3way_guardrail,passes_pure_test_guardrail,pure_test_guardrail_reason
0,optuna_object_matrix_0149,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,1,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_chi2_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.509259,0.000000,0.437500,0.781250,0.000000,0.000000,0.597403,0.402597,True,True,True,pass
1,optuna_object_matrix_0081,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,2,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,data_driven_emp_cv,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.546296,0.000000,0.437500,0.781250,0.000000,0.000000,0.610390,0.389610,True,True,True,pass
2,optuna_object_matrix_0148,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,3,object_matrix,object_median,rule_variant_grid,object_median,sg_smooth,alternative_empHQ_fixed2,5,0.01,0.50,0.50,0.95,0.000000,0.927273,0.000000,0.601852,0.000000,0.437500,0.781250,0.000000,0.000000,0.649351,0.350649,True,True,True,pass
3,optuna_object_matrix_0184,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,4,object_matrix,object_median,rule_variant_grid,object_median,absorbance_sg_smooth,simple_emp_cv,8,0.01,0.50,0.30,0.70,0.018868,0.527273,0.000000,0.564815,0.034483,0.562500,0.701509,0.000000,0.229167,0.454545,0.545455,True,True,True,pass
4,optuna_object_matrix_0152,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,5,object_matrix,object_median,rule_variant_grid,object_median,absorbance_sg_d2,alternative_empHQ_emp_cv,7,0.01,0.50,0.25,0.65,0.150943,0.418182,0.000000,0.500000,0.034483,0.562500,0.701509,0.000000,0.354167,0.363636,0.636364,True,True,True,pass
5,optuna_object_matrix_0105,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,7,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,3,0.01,0.55,0.15,0.55,0.264151,0.000000,0.000000,0.416667,0.344828,0.062500,0.796336,0.000000,0.062500,0.480519,0.519481,True,True,True,pass
6,optuna_object_matrix_0121,04B2_optuna_challenge,04B2_optuna_binary_pareto__3way_calibrated,8,object_matrix,object_mean,rule_variant_grid,object_mean,snv_sg_smooth,alternative_empHQ_fixed2,3,0.01,0.55,0.15,0.55,0.264151,0.000000,0.000000,0.416667,0.344828,0.062500,0.796336,0.000000,0.062500,0.480519,0.519481,True,True,True,pass
7,04A_object_matrix_0002,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,10,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,data_driven_emp_cv,<NA>,0.01,0.70,0.55,0.95,0.018868,0.781818,0.000000,0.712963,0.103448,0.541667,0.677443,0.034483,0.000000,0.857143,0.142857,True,True,True,pass
8,04A_object_matrix_0003,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,11,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,data_driven_emp_cv,<NA>,0.01,0.75,0.55,0.95,0.056604,0.745455,0.000000,0.712963,0.310345,0.437500,0.626078,0.034483,0.000000,0.857143,0.142857,True,True,True,pass
9,04A_object_matrix_0004,04B_grid_robustness,04A_grid_rule_variant_universe__04B_robust_pareto,12,object_matrix,object_median,empirical_cv_rule,object_median,absorbance_sg_d2,simple_emp_cv,<NA>,0.01,0.70,0.55,0.95,0.075472,0.672727,0.000000,0.814815,0.241379,0.500000,0.629310,0.034483,0.000000,0.818182,0.181818,True,True,True,

In [13]:
plot_validation_test_shift(
    pure_test_model_summary_df,
    validation_metric="validation_fn_rate",
    test_metric="pure_test_fn_rate",
    color_col="matrix_family",
    id_col="selected_config_id",
    annotate_top_n=5,
    title="Validation batch 3 vs pure test batch 4 — FN rate",
    show=True,
)

plot_validation_test_shift(
    pure_test_model_summary_df,
    validation_metric="validation_fp_rate",
    test_metric="pure_test_fp_rate",
    color_col="matrix_family",
    id_col="selected_config_id",
    annotate_top_n=5,
    title="Validation batch 3 vs pure test batch 4 — FP rate",
    show=True,
)

## 7. Optional border/core diagnostic on pure test

This diagnostic checks whether test errors are mostly concentrated on object borders or in object cores. It should not be used to retune the current models in this notebook.


In [14]:
if RUN_BORDER_DIAGNOSTIC:
    border_config_ids = (
        pure_test_model_summary_df
        .sort_values(
            ["passes_pure_test_guardrail", "matrix_family", "candidate_source", "candidate_input_rank"],
            ascending=[False, True, True, True],
        )
        .groupby(["matrix_family", "candidate_source"], group_keys=False, dropna=False)
        .head(max(1, BORDER_DIAGNOSTIC_CONFIG_LIMIT // 4))
        ["selected_config_id"]
        .astype(str)
        .tolist()
    )

    pure_test_pixels_for_border_df = pure_test_pixels_df[
        pure_test_pixels_df["selected_config_id"].astype(str).isin(border_config_ids)
    ].copy()

    pure_test_border_diagnostic_df = summarize_border_diagnostics_by_config(
        pixel_df=pure_test_pixels_for_border_df,
        object_db=object_db,
        target_class=TARGET_CLASS,
        border_widths=BORDER_DIAGNOSTIC_WIDTHS,
        config_cols=[
            "selected_config_id",
            "candidate_source",
            "matrix_family",
            "training_matrix_id",
            "matrix_method",
            "preprocessing",
            "selected_rule_name",
            "n_components",
            "alpha",
            "object_threshold",
        ],
    )

else:
    pure_test_border_diagnostic_df = pd.DataFrame()

save_parquet_if_nonempty(
    pure_test_border_diagnostic_df,
    PURE_TEST_BORDER_DIAGNOSTIC_PATH,
)

print("Pure test border diagnostic:", pure_test_border_diagnostic_df.shape)

display(pure_test_border_diagnostic_df.head(80))


Pure test border diagnostic: (150, 22)


,selected_config_id,candidate_source,matrix_family,training_matrix_id,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,border_width,zone,n_pixels,tp,tn,fp,fn,n_errors,error_rate,fp_rate,fn_rate,pixel_accuracy
0,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,1,border,2084,113,1227,54,690,744,0.357006,0.042155,0.859278,0.642994
1,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,1,core,6317,1133,3677,279,1228,1507,0.238563,0.070526,0.520119,0.761437
2,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,2,border,4145,466,2362,180,1137,1317,0.317732,0.070810,0.709295,0.682268
3,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,2,core,4256,780,2542,153,781,934,0.219455,0.056772,0.500320,0.780545
4,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,3,border,6323,906,3614,276,1527,1803,0.285149,0.070951,0.627620,0.714851
5,04A_object_matrix_0001,04B_grid_robustness,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,NaN,0.01,0.75,3,core,2078,340,1290,57,391,448,0.215592,0.042316,0.534884,0.784408
6,04A_object_matrix_0002,04B_grid_robustness,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,1,border,2084,510,570,711,293,1004,0.481766,0.555035,0.364882,0.518234
7,04A_object_matrix_0002,04B_grid_robustness,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,1,core,6317,2039,1085,2871,322,3193,0.505461,0.725733,0.136383,0.494539
8,04A_object_matrix_0002,04B_grid_robustness,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,2,border,4145,1172,939,1603,431,2034,0.490712,0.630606,0.268871,0.509288
9,04A_object_matrix_0002,04B_grid_robustness,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,NaN,0.01,0.70,2,core,4256,1377,716,1979,184,2163,0.508224,0.734323,0.117873,0.491776


In [15]:
if len(pure_test_border_diagnostic_df) > 0:
    plot_border_core_metrics(
        pure_test_border_diagnostic_df,
        border_width_col="border_width",
        zone_col="zone",
        metric_cols=(
            "fn_rate",
            "fp_rate",
            "pixel_accuracy",
        ),
        config_col="selected_config_id",
        title="Pure test — border versus core diagnostics",
        show=True,
    )

## 8. Freeze reference configurations for mixture application

The pure test is an external check. The selection below primarily preserves the previous candidate order and uses batch 4 only as a guardrail against major failures.


In [16]:
# ---------------------------------------------------------------------
# Freeze reference configurations for mixture application
# ---------------------------------------------------------------------
# Batch 4 is used as an external guardrail only.
# The ranking from 04B/04B2 is preserved.

eligible_df = pure_test_model_summary_df.copy()

if USE_TEST_GUARDRAILS:
    eligible_parts = []

    grouping_cols = ["matrix_family"]
    if PRESERVE_CANDIDATE_SOURCES and "candidate_source" in eligible_df.columns:
        grouping_cols.append("candidate_source")

    for key, group in eligible_df.groupby(grouping_cols, dropna=False):
        passed = group[group["passes_pure_test_guardrail"].astype(bool)].copy()

        if len(passed) > 0:
            eligible_parts.append(passed)
        else:
            print(
                "[WARNING] No candidate passed pure-test guardrails for group:",
                key,
                "Keeping this group for inspection."
            )
            eligible_parts.append(group.copy())

    eligible_df = pd.concat(
        eligible_parts,
        ignore_index=True,
        sort=False,
    )

eligible_df = (
    eligible_df
    .sort_values(
        [
            "matrix_family",
            "candidate_source",
            "candidate_input_rank",
        ],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

if PRESERVE_CANDIDATE_SOURCES and "candidate_source" in eligible_df.columns:
    frozen_reference_configs_df = (
        eligible_df
        .groupby(["matrix_family", "candidate_source"], group_keys=False, dropna=False)
        .head(N_FROZEN_PER_MATRIX_FAMILY_SOURCE)
        .sort_values(["matrix_family", "candidate_source", "candidate_input_rank"])
        .groupby("matrix_family", group_keys=False, dropna=False)
        .head(N_FROZEN_PER_MATRIX_FAMILY)
        .copy()
        .reset_index(drop=True)
    )
else:
    frozen_reference_configs_df = (
        eligible_df
        .groupby("matrix_family", group_keys=False, dropna=False)
        .head(N_FROZEN_PER_MATRIX_FAMILY)
        .copy()
        .reset_index(drop=True)
    )

if N_FROZEN_OVERALL is not None:
    print(
        "[WARNING] N_FROZEN_OVERALL is active. "
        "This can unbalance matrix families or candidate sources."
    )
    frozen_reference_configs_df = (
        frozen_reference_configs_df
        .head(int(N_FROZEN_OVERALL))
        .copy()
        .reset_index(drop=True)
    )

frozen_reference_configs_df["frozen_reference_rank"] = np.arange(
    1,
    len(frozen_reference_configs_df) + 1,
)

frozen_reference_configs_df["selection_strategy"] = (
    frozen_reference_configs_df["selection_strategy"].astype(str)
    + "__04C_pure_test_checked"
)

frozen_reference_configs_df["ready_for_mixture_application"] = True


CONFIG_COLS = [
    "selected_config_id",
    "frozen_reference_rank",
    "ready_for_mixture_application",

    "candidate_source",
    "candidate_source_file",
    "candidate_file_source_kind",
    "candidate_input_rank",

    "selection_split",
    "selection_strategy",

    "model_family",
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
    "m",
    "m_effective",

    "preprocessing",
    "preprocessing_steps",

    "rule",
    "rule_variant",
    "selected_rule_name",
    "rule_for_refit",
    "limit_source",

    "target_class",
    "non_target_label",

    "n_components",
    "alpha",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",

    # Validation binary metrics.
    "validation_balanced_accuracy",
    "validation_target_sensitivity",
    "validation_non_target_specificity",
    "validation_fn_rate",
    "validation_fp_rate",

    # Validation 3-way metrics.
    "validation_3way_target_miss_rate",
    "validation_3way_screening_sensitivity",
    "validation_3way_non_target_false_accept_rate",
    "validation_3way_uncertain_rate",
    "validation_3way_coverage_rate",

    # Robustness metadata from 04B.
    "mean_fn_rate",
    "std_fn_rate",
    "max_fn_rate",
    "mean_fp_rate",
    "std_fp_rate",
    "max_fp_rate",
    "mean_balanced_accuracy",
    "is_robust_2way_pareto",
    "is_robust_3way_pareto",
    "robust_pareto_axis",
    "passes_robustness_filters",

    # Optuna metadata.
    "optuna_trial_number",
    "value_0",
    "value_1",
    "value_2",
    "objective_fn_rate_max",
    "objective_fp_rate_mean",
    "objective_balanced_accuracy_mean",
    "fn_rate_max",
    "fn_rate_mean",
    "fn_rate_std",
    "fp_rate_mean",
    "fp_rate_max",
    "fp_rate_std",
    "balanced_accuracy_mean",
    "object_threshold_median",

    # Pure test binary metrics.
    "pure_test_n",
    "pure_test_tp",
    "pure_test_fn",
    "pure_test_fp",
    "pure_test_tn",
    "pure_test_balanced_accuracy",
    "pure_test_target_sensitivity",
    "pure_test_non_target_specificity",
    "pure_test_fn_rate",
    "pure_test_fp_rate",
    "pure_test_f1_score",
    "pure_test_accuracy",
    "pure_test_precision",
    "pure_test_selection_score",

    # Pure test 3-way metrics.
    "pure_test_3way_n",
    "pure_test_3way_n_target",
    "pure_test_3way_n_non_target",
    "pure_test_3way_target_miss_rate",
    "pure_test_3way_screening_sensitivity",
    "pure_test_3way_non_target_false_accept_rate",
    "pure_test_3way_uncertain_rate",
    "pure_test_3way_coverage_rate",
    "pure_test_3way_non_target_auto_reject_rate",
    "pure_test_3way_decided_balanced_accuracy",

    # Guardrails.
    "passes_pure_test_binary_guardrail",
    "passes_pure_test_3way_guardrail",
    "passes_pure_test_guardrail",
    "pure_test_guardrail_reason",
]

CONFIG_COLS = [
    col for col in CONFIG_COLS
    if col in frozen_reference_configs_df.columns
]

frozen_reference_configs_df = frozen_reference_configs_df[CONFIG_COLS].copy()

save_parquet(frozen_reference_configs_df, FROZEN_REFERENCE_CONFIGS_PATH)

print("Frozen reference configs:", frozen_reference_configs_df.shape)
print("Saved:", FROZEN_REFERENCE_CONFIGS_PATH)

display(frozen_reference_configs_df)

Frozen reference configs: (16, 97)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\frozen_reference_configs.parquet


,selected_config_id,frozen_reference_rank,ready_for_mixture_application,candidate_source,candidate_source_file,candidate_file_source_kind,candidate_input_rank,selection_split,selection_strategy,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,n_components,alpha,object_threshold,three_way_lower_threshold,three_way_upper_threshold,sg_window_length,sg_polyorder,position_dilation_radius,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,is_robust_2way_pareto,is_robust_3way_pareto,robust_pareto_axis,optuna_trial_number,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,fn_rate_max,fn_rate_mean,fn_rate_std,fp_rate_mean,fp_rate_max,fp_rate_std,balanced_accuracy_mean,object_threshold_median,pure_test_n,pure_test_tp,pure_test_fn,pure_test_fp,pure_test_tn,pure_test_balanced_accuracy,pure_test_target_sensitivity,pure_test_non_target_specificity,pure_test_fn_rate,pure_test_fp_rate,pure_test_f1_score,pure_test_accuracy,pure_test_precision,pure_test_selection_score,pure_test_3way_n,pure_test_3way_n_target,pure_test_3way_n_non_target,pure_test_3way_target_miss_rate,pure_test_3way_screening_sensitivity,pure_test_3way_non_target_false_accept_rate,pure_test_3way_uncertain_rate,pure_test_3way_coverage_rate,pure_test_3way_non_target_auto_reject_rate,pure_test_3way_decided_balanced_accuracy,passes_pure_test_binary_guardrail,passes_pure_test_3way_guardrail,passes_pure_test_guardrail,pure_test_guardrail_reason
0,optuna_object_matrix_0149,1,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,1,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,peanut,almond,5,0.01,0.50,0.50,0.95,15,2,4,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.345455,0.509259,0.490741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,149.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.536364,0.50,77,29,0,21,27,0.781250,1.000000,0.562500,0.000000,0.437500,0.734177,0.727273,0.580000,-0.386246,77,29,48,0.000000,1.000000,0.000000,0.597403,0.402597,0.562500,1.000000,True,True,True,pass
1,optuna_object_matrix_0081,2,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,2,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,object_matrix,object_median,object_median,random,NaN,40,NaN,sg_smooth,sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,5,0.01,0.50,0.50,0.95,15,2,5,0.536364,1.000000,0.072727,0.000000,0.927273,0.0,1.0,0.290909,0.546296,0.453704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,81.0,0.000000,0.927273,0.536364,0.000000,0.927273,0.536364,0.000000,0.000000,0.000000e+00,0.927273,0.927273,0.000000e+00,0.536364,0.50,77,29,0,21,27,0.781250,1.000000,0.562500,0.000000,0.437500,0.734177,0.727273,0.580000,-0.386246,77,29,48,0.000000,1.000000,0.000000,0.610390,0.389610,0.562500,1.000000,True,True,True,pass
2,optuna_object_matrix_0148,3,True,04B2_optuna_challenge,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,3,validation_batch_3,04B2_optuna_binary_pareto__3way_calibrated__04...,rule_variant_grid,ob

In [17]:
if len(pure_test_border_diagnostic_df) > 0:
    plot_border_core_metrics(
        pure_test_border_diagnostic_df,
        border_width_col="border_width",
        zone_col="zone",
        metric_cols=(
            "fn_rate",
            "fp_rate",
            "pixel_accuracy",
        ),
        config_col="selected_config_id",
        title="Pure test — border versus core diagnostics",
        show=True,
    )

## 9. Protocol summary

In [18]:
pure_test_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),
    "candidate_configs_path": str(CANDIDATE_CONFIGS_PATH),
    "candidate_source": CANDIDATE_SOURCE,

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)),

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "pure_test_train_batches_json": json.dumps(PURE_TEST_TRAIN_BATCHES),
    "pure_test_train_filters_json": json.dumps(PURE_TEST_TRAIN_FILTERS, default=str),
    "pure_test_filters_json": json.dumps(PURE_TEST_FILTERS, default=str),

    "random_state": int(RANDOM_STATE),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "cv_n_splits": int(CV_N_SPLITS) if CV_N_SPLITS is not None else np.nan,
    "cv_group_col": CV_GROUP_COL,

    "use_test_guardrails": bool(USE_TEST_GUARDRAILS),
    "max_pure_test_binary_fn_rate": float(MAX_PURE_TEST_BINARY_FN_RATE),
    "max_pure_test_binary_fp_rate": float(MAX_PURE_TEST_BINARY_FP_RATE),
    "min_pure_test_binary_balanced_accuracy": float(MIN_PURE_TEST_BINARY_BALANCED_ACCURACY),

    "max_pure_test_3way_target_miss_rate": float(MAX_PURE_TEST_3WAY_TARGET_MISS_RATE),
    "max_pure_test_3way_false_accept_rate": float(MAX_PURE_TEST_3WAY_FALSE_ACCEPT_RATE),
    "max_pure_test_3way_uncertain_rate": float(MAX_PURE_TEST_3WAY_UNCERTAIN_RATE),

    "preserve_candidate_sources": bool(PRESERVE_CANDIDATE_SOURCES),
    "n_frozen_per_matrix_family_source": int(N_FROZEN_PER_MATRIX_FAMILY_SOURCE),
    "n_frozen_overall": (
        int(N_FROZEN_OVERALL)
        if N_FROZEN_OVERALL is not None
        else np.nan
    ),

    "run_border_diagnostic": bool(RUN_BORDER_DIAGNOSTIC),
    "border_diagnostic_widths_json": json.dumps(BORDER_DIAGNOSTIC_WIDTHS),
    "border_diagnostic_config_limit": int(BORDER_DIAGNOSTIC_CONFIG_LIMIT),

    "n_candidate_configs": int(len(candidate_configs_df)),
    "n_pure_test_metrics": int(len(pure_test_metrics_df)),
    "n_pure_test_objects": int(len(pure_test_objects_df)),
    "n_pure_test_3way_metrics": int(len(pure_test_3way_metrics_df)),
    "n_pure_test_3way_objects": int(len(pure_test_3way_objects_df)),
    "n_pure_test_3way_by_image": int(len(pure_test_3way_by_image_df)),
    "n_pure_test_pixel_error_rows": int(len(pure_test_pixel_errors_by_image_df)),
    "n_pure_test_refit_errors": int(len(pure_test_refit_errors_df)),
    "n_frozen_reference_configs": int(len(frozen_reference_configs_df)),

    "pure_test_metrics_path": str(PURE_TEST_METRICS_PATH),
    "pure_test_model_summary_path": str(PURE_TEST_MODEL_SUMMARY_PATH),
    "pure_test_object_predictions_path": str(PURE_TEST_OBJECT_PREDICTIONS_PATH),
    "pure_test_object_errors_by_image_path": str(PURE_TEST_OBJECT_ERRORS_BY_IMAGE_PATH),
    "pure_test_3way_metrics_path": str(PURE_TEST_3WAY_METRICS_PATH),
    "pure_test_3way_objects_path": str(PURE_TEST_3WAY_OBJECTS_PATH),
    "pure_test_3way_by_image_path": str(PURE_TEST_3WAY_BY_IMAGE_PATH),
    "pure_test_pixel_errors_by_image_path": str(PURE_TEST_PIXEL_ERRORS_BY_IMAGE_PATH),
    "pure_test_border_diagnostic_path": str(PURE_TEST_BORDER_DIAGNOSTIC_PATH),
    "frozen_reference_configs_path": str(FROZEN_REFERENCE_CONFIGS_PATH),

    "candidate_sources_json": json.dumps(
        candidate_configs_df["candidate_source"]
        .value_counts(dropna=False)
        .astype(int)
        .to_dict(),
        default=str,
    ),

    "frozen_candidate_sources_json": json.dumps(
        frozen_reference_configs_df["candidate_source"]
        .value_counts(dropna=False)
        .astype(int)
        .to_dict(),
        default=str,
    ),
}])

save_parquet(pure_test_protocol_df, PURE_TEST_PROTOCOL_PATH)

print("Saved pure test protocol:")
print(PURE_TEST_PROTOCOL_PATH)

display(pure_test_protocol_df)

Saved pure test protocol:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_protocol.parquet


,db_h5_path,results_dir,candidate_configs_path,candidate_source,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,non_target_label,reference_classes_json,pure_test_train_batches_json,pure_test_train_filters_json,pure_test_filters_json,random_state,replace_balanced_pixels,cv_n_splits,cv_group_col,use_test_guardrails,max_pure_test_binary_fn_rate,max_pure_test_binary_fp_rate,min_pure_test_binary_balanced_accuracy,max_pure_test_3way_target_miss_rate,max_pure_test_3way_false_accept_rate,max_pure_test_3way_uncertain_rate,preserve_candidate_sources,n_frozen_per_matrix_family_source,n_frozen_overall,run_border_diagnostic,border_diagnostic_widths_json,border_diagnostic_config_limit,n_candidate_configs,n_pure_test_metrics,n_pure_test_objects,n_pure_test_3way_metrics,n_pure_test_3way_objects,n_pure_test_3way_by_image,n_pure_test_pixel_error_rows,n_pure_test_refit_errors,n_frozen_reference_configs,pure_test_metrics_path,pure_test_model_summary_path,pure_test_object_predictions_path,pure_test_object_errors_by_image_path,pure_test_3way_metrics_path,pure_test_3way_objects_path,pure_test_3way_by_image_path,pure_test_pixel_errors_by_image_path,pure_test_border_diagnostic_path,frozen_reference_configs_path,candidate_sources_json,frozen_candidate_sources_json
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,04B2_grid_plus_optuna,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,almond,"[""almond"", ""peanut""]","[1, 2, 3]","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...",42,False,5,object_id,True,0.35,0.8,0.5,0.35,0.8,0.9,True,4,NaN,True,"[1, 2, 3]",30,31,31,2387,31,2387,62,62,0,16,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,"{""04B_grid_robustness"": 18, ""04B2_optuna_chall...","{""04B2_optuna_challenge"": 8, ""04B_grid_robustn..."


## 10. Final check

In [19]:
print("04C_simca_pure_test_evaluation.ipynb completed.")
print()
print("Essential outputs:")
print(" -", PURE_TEST_METRICS_PATH)
print(" -", PURE_TEST_MODEL_SUMMARY_PATH)
print(" -", PURE_TEST_OBJECT_PREDICTIONS_PATH)
print(" -", PURE_TEST_OBJECT_ERRORS_BY_IMAGE_PATH)
print(" -", PURE_TEST_PIXEL_ERRORS_BY_IMAGE_PATH)
print(" -", PURE_TEST_3WAY_METRICS_PATH)
print(" -", PURE_TEST_3WAY_OBJECTS_PATH)
print(" -", PURE_TEST_3WAY_BY_IMAGE_PATH)
print(" -", FROZEN_REFERENCE_CONFIGS_PATH)
print(" -", PURE_TEST_PROTOCOL_PATH)

if pure_test_border_diagnostic_df is not None and len(pure_test_border_diagnostic_df) > 0:
    print(" -", PURE_TEST_BORDER_DIAGNOSTIC_PATH)

if pure_test_refit_errors_df is not None and len(pure_test_refit_errors_df) > 0:
    print(" -", PURE_TEST_REFIT_ERRORS_PATH)

if SAVE_PURE_TEST_PIXEL_TABLE:
    print(" -", PURE_TEST_PIXEL_PREDICTIONS_PATH)

print()
print("Summary:")
print(f" - Candidate source: {CANDIDATE_SOURCE}")
print(f" - Candidate configs tested: {len(candidate_configs_df)}")
print(f" - Pure test metric rows: {len(pure_test_metrics_df)}")
print(f" - Pure test 3-way metric rows: {len(pure_test_3way_metrics_df)}")
print(f" - Pure test 3-way object rows: {len(pure_test_3way_objects_df)}")
print(f" - Frozen reference configs: {len(frozen_reference_configs_df)}")
print(f" - Refit errors: {len(pure_test_refit_errors_df)}")
print()
print("Next notebook:")
print("05_simca_mixture_application.ipynb")

print()
print("Result files:")
display(list_result_files(RESULTS_DIR).head(30))


04C_simca_pure_test_evaluation.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_model_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_object_predictions.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_object_errors_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_pixel_errors_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_test_non_noisy_all\pure_test_3way_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_pure_te

,file,suffixes,size_mb
0,pure_test_3way_objects.parquet,.parquet,0.102088
1,pure_test_object_predictions.parquet,.parquet,0.101136
2,pure_test_model_summary.parquet,.parquet,0.093671
3,pure_test_metrics.parquet,.parquet,0.074702
4,frozen_reference_configs.parquet,.parquet,0.063258
5,pure_test_protocol.parquet,.parquet,0.043014
6,pure_test_refit_errors.parquet,.parquet,0.040366
7,pure_test_pixel_errors_by_image.parquet,.parquet,0.027691
8,pure_test_object_errors_by_image.parquet,.parquet,0.026995
9,pure_test_border_diagnostic.parquet,.parquet,0.017983
